In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. Synthetic Data Generation (y = 2x + 3)
X = np.random.rand(100, 1).astype(np.float32)
y = 2 * X + 3 + np.random.randn(100, 1).astype(np.float32) * 0.1

X_tensor = torch.from_numpy(X)
y_tensor = torch.from_numpy(y)

# 2. Linear Regression Model
model = nn.Linear(1, 1)

# 3. Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 4. Training Loop
for epoch in range(100):
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}")

print(f"Trained Weight: {model.weight.item():.4f}, Bias: {model.bias.item():.4f}")

Epoch [20/100], Loss: 0.0143
Epoch [40/100], Loss: 0.0129
Epoch [60/100], Loss: 0.0122
Epoch [80/100], Loss: 0.0118
Epoch [100/100], Loss: 0.0115
Trained Weight: 1.9358, Bias: 3.0214


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Neural Network with Dropout
class DropoutNet(nn.Module):
    def __init__(self):
        super(DropoutNet, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

# 2. Data Loader
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_loader = DataLoader(datasets.MNIST(root='./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)

# 3. Initialize & Train 1 Epoch Demo
model = DropoutNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train() # Enable Dropout
for batch_idx, (data, target) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    if batch_idx % 200 == 0:
        print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")
    if batch_idx > 400: break # Quick demo limit

Batch 0, Loss: 2.4192


Epoch [10/50], Loss: 0.7630
Epoch [20/50], Loss: 0.6246
Epoch [30/50], Loss: 0.5329
Epoch [40/50], Loss: 0.4698
Epoch [50/50], Loss: 0.4241


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Simple shared model setup comparison framework
X = torch.randn(100, 10)
y = torch.randint(0, 2, (100,))

model_adam = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 2))
model_sgd = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 2))

# Share initial state weights for fairness
model_sgd.load_state_dict(model_adam.state_dict())

opt_adam = optim.Adam(model_adam.parameters(), lr=0.01)
opt_sgd = optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

# Single Step Update Comparison Demo
loss_a = criterion(model_adam(X), y)
opt_adam.zero_grad()
loss_a.backward()
opt_adam.step()

loss_s = criterion(model_sgd(X), y)
opt_sgd.zero_grad()
loss_s.backward()
opt_sgd.step()

print(f"Initial Adam step loss: {loss_a.item():.4f}")
print(f"Initial SGD step loss: {loss_s.item():.4f}")

Initial Adam step loss: 0.6895
Initial SGD step loss: 0.6895


In [1]:
import torch
import torch.nn as nn

# 1. Custom Element-wise Scale Layer
class ElementwiseScaleLayer(nn.Module):
    def __init__(self, num_features):
        super(ElementwiseScaleLayer, self).__init__()
        # Learnable parameter
        self.weights = nn.Parameter(torch.ones(num_features))
        
    def forward(self, x):
        return x * self.weights

# 2. Testing Custom Model Layer Components
model = nn.Sequential(
    nn.Linear(5, 5),
    ElementwiseScaleLayer(5),
    nn.ReLU()
)

sample_input = torch.randn(2, 5)
output = model(sample_input)
print("Input shape:", sample_input.shape)
print("Output tensor sample:\n", output)

Input shape: torch.Size([2, 5])
Output tensor sample:
 tensor([[0.0000, 0.0000, 0.0000, 0.6626, 0.2222],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]], grad_fn=<ReluBackward0>)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# 1. Model & Component Pipeline
model = nn.Linear(10, 2)
optimizer = optim.SGD(model.parameters(), lr=0.1)
# Decay LR by a factor of 0.1 every 5 epochs
scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

# 2. Simulate step tracking over epochs
print("Simulating step adjustments:")
for epoch in range(12):
    # Dummy step update process
    optimizer.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1:02d} -> Active Learning Rate: {current_lr:.5f}")
    scheduler.step()

Simulating step adjustments:
Epoch 01 -> Active Learning Rate: 0.10000
Epoch 02 -> Active Learning Rate: 0.10000
Epoch 03 -> Active Learning Rate: 0.10000
Epoch 04 -> Active Learning Rate: 0.10000
Epoch 05 -> Active Learning Rate: 0.10000
Epoch 06 -> Active Learning Rate: 0.01000
Epoch 07 -> Active Learning Rate: 0.01000
Epoch 08 -> Active Learning Rate: 0.01000
Epoch 09 -> Active Learning Rate: 0.01000
Epoch 10 -> Active Learning Rate: 0.01000
Epoch 11 -> Active Learning Rate: 0.00100
Epoch 12 -> Active Learning Rate: 0.00100
